<a href="https://colab.research.google.com/github/IgnBys/3D_splatting/blob/main/GS_VTON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/yukangcao/GS-VTON.git


Cloning into 'GS-VTON'...
remote: Enumerating objects: 1439, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 1439 (delta 2), reused 0 (delta 0), pack-reused 1431 (from 1)
Receiving objects: 100% (1439/1439), 163.54 MiB | 25.56 MiB/s, done.
Resolving deltas: 100% (512/512), done.


### Step 1: Install PyTorch 2.2.1 and xformers 0.0.25 (CUDA 11.8)

In [ ]:
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu118
!pip install xformers==0.0.25 --no-deps --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.1/819.1 MB 751.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 73.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 49.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 43.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 972.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204

### Step 2: Install Additional Dependencies
This includes IP-Adapter, BasicSR, and Detectron2.

In [1]:
!pip install git+https://github.com/tencent-ailab/IP-Adapter.git
!pip install git+https://github.com/XPixelGroup/BasicSR@8d56e3a045f9fb3e1d8872f92ee4a4f07f886b0a
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/tencent-ailab/IP-Adapter.git to /tmp/pip-req-build-qddeogya
  Running command git clone --filter=blob:none --quiet https://github.com/tencent-ailab/IP-Adapter.git /tmp/pip-req-build-qddeogya
  Resolved https://github.com/tencent-ailab/IP-Adapter.git to commit 62e4af9d0c1ac7d5f8dd386a0ccf2211346af1a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ip-adapter: filename=ip_adapter-0.1.0-py3-none-any.whl size=32730 sha256=4e334c6a361307831a5ed4271c9356dfe6ada8b5fa4cf187256a22273713e81f
  Stored in directory: /tmp/pip-ephem-wheel-cache-unkhgvba/wheels/96/32/4d/f703ddfd0794868b989a766f963f572268b241063dba5ecddb
Successfully built ip-adapter
  Cloning https://github.com/XPixelGroup/BasicSR (to revision 8d56e3a045f9fb3e1d8872f92ee4a4f07f886b0a) to /tmp/pip-req-build-ih63mrto
  Running command git clone --filter=blob:none --quiet https://github.com/XPixelGroup/

### Step 3: Install GS-VTON Requirements
Navigating into the cloned directory and installing the `requirements.txt`.

In [3]:
import os

# Ensure we are in the correct directory
if not os.path.exists('/content/GS-VTON'):
    !git clone https://github.com/yukangcao/GS-VTON.git

%cd /content/GS-VTON

# 1. Setup CUDA environment strictly for compilation
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] = '/usr/local/cuda-11.8/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

# 2. Update submodules manually
!git submodule update --init --recursive

# 3. Install build dependencies first
!pip install ninja setuptools wheel numpy

# 4. Install nvdiffrast manually to bypass build isolation
if not os.path.exists('/content/nvdiffrast'):
    !git clone https://github.com/NVlabs/nvdiffrast.git /content/nvdiffrast
%cd /content/nvdiffrast
!pip install . --no-build-isolation

# 5. Return to GS-VTON and install Gaussian Splatting submodules
%cd /content/GS-VTON
if os.path.exists('submodules/diff-gaussian-rasterization'):
    !pip install ./submodules/diff-gaussian-rasterization --no-build-isolation
if os.path.exists('submodules/simple-knn'):
    !pip install ./submodules/simple-knn --no-build-isolation

# 6. Clean requirements and install the rest
!sed -i '/nvdiffrast/d' requirements.txt
!sed -i '/diff-gaussian-rasterization/d' requirements.txt
!sed -i '/simple-knn/d' requirements.txt
!pip install -r requirements.txt

/content/GS-VTON
/content/nvdiffrast
Processing /content/nvdiffrast
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for nvdiffrast (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for nvdiffrast
Failed to build nvdiffrast
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (nvdiffrast)
/content/GS-VTON
  Cloning https://github.com/ashawkey/envlight.git to /tmp/pip-req-build-2pz4uckg
  Running command git clone --filter=blob:none --quiet https://github.com/ashawkey/envlight.git /tmp/pip-req-build-2pz4uckg
  Resolved https://github.com/ashawkey/envlight.git to commit 05b5851e854429d72ecaf5b206ed64ce55fae677
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-p_wa1hux
  Running comma

In [ ]:
# 1. Ensure CUDA 11.8 environment is active
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda-11.8'
os.environ['PATH'] = '/usr/local/cuda-11.8/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

# 2. Verify/Re-install nvdiffrast
if not os.path.exists('/content/nvdiffrast'):
    !git clone https://github.com/NVlabs/nvdiffrast.git /content/nvdiffrast
%cd /content/nvdiffrast
!find . -name "*.h" -o -name "*.cpp" -o -name "*.cu" | xargs sed -i '1i #include <cstdint>'
!pip install -e . --no-build-isolation

# 3. Install Gaussian Splatting submodules
sub_root = '/content/GS-VTON/GS-VTON/stage2/gaussiansplatting/submodules'

# Install diff-gaussian-rasterization
if os.path.exists(f'{sub_root}/diff-gaussian-rasterization'):
    print('\n--- Installing diff-gaussian-rasterization ---')
    %cd {sub_root}/diff-gaussian-rasterization
    !rm -rf build dist *.egg-info
    !find . -name "*.h" -o -name "*.cpp" -o -name "*.cu" | xargs sed -i '1i #include <cstdint>'
    !pip install -e . --no-build-isolation

# Install simple-knn
if os.path.exists(f'{sub_root}/simple-knn'):
    print('\n--- Installing simple-knn ---')
    %cd {sub_root}/simple-knn
    !rm -rf build dist *.egg-info
    !pip install -e . --no-build-isolation

# 4. Final Verification
%cd /content/GS-VTON
try:
    import nvdiffrast
    import diff_gaussian_rasterization
    import simple_knn
    print('\nVerification SUCCESS: Core modules are ready.')
except Exception as e:
    print(f'\nVerification FAILED: {e}')

/content/nvdiffrast
Obtaining file:///content/nvdiffrast
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
